# 🚀 OmniTriageEnv — GRPO Training Notebook

**Train an LLM to triage customer emails using Reinforcement Learning**

This notebook trains a Llama-3.2-1B model using GRPO (Group Relative Policy Optimization) on the OmniTriageEnv environment. The model learns to:
- Classify email priority (urgent / normal / low)
- Route to the correct department (billing / technical / returns / general)
- Decide when to escalate to a human agent
- Draft professional responses with relevant keywords

**Requirements:** T4 GPU (free Colab), ~20 minutes for 200 steps

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---

## 1️⃣ Install Dependencies

In [ ]:
%%capture
!pip install unsloth trl datasets matplotlib pydantic fastapi
# Unsloth provides 2x faster training with 60% less memory on T4 GPUs

## 2️⃣ Clone the OmniTriageEnv Environment

In [ ]:
!git clone https://huggingface.co/spaces/Prakhar132/email-triage-env OmniTriageEnv
%cd OmniTriageEnv
!ls -la *.py

## 3️⃣ Quick Environment Sanity Check
Verify the environment works before training.

In [ ]:
import sys, os
sys.path.insert(0, '.')

from emails import EMAILS, GROUND_TRUTH, TASK_EMAIL_IDS
from environment import OmniTriageEnv

# Quick test
env = OmniTriageEnv()
obs = env.reset(difficulty='easy')
print(f'✅ Environment loaded: {obs.queue_length} emails in queue')
print(f'📧 First email: {obs.current_email.subject}')
print(f'📊 Total emails in corpus: {len(EMAILS)}')
print(f'📋 Ground truth entries: {len(GROUND_TRUTH)}')
print(f'\nDifficulty levels: {list(TASK_EMAIL_IDS.keys())}')

## 4️⃣ Run GRPO Training

This is the main training loop. It:
1. Builds prompts from the 30-email corpus
2. Loads Llama-3.2-1B with Unsloth (4-bit LoRA)
3. Runs baseline evaluation (untrained model)
4. Trains with GRPO for 200 steps
5. Runs post-training evaluation
6. Generates reward curves and before/after comparison plots

In [ ]:
!python train_grpo.py \
    --model unsloth/Llama-3.2-1B-Instruct \
    --steps 200 \
    --batch-size 4 \
    --num-generations 4 \
    --lr 5e-6 \
    --output-dir ./training_output

## 5️⃣ View Training Results

### Reward Curve
Shows how the model's reward improves over training steps.

In [ ]:
from IPython.display import Image, display
import os

reward_path = './training_output/reward_curve.png'
if os.path.exists(reward_path):
    display(Image(reward_path, width=800))
    print('📈 Reward curve: higher = model is learning better triage')
else:
    print('⚠️ Reward curve not found — training may not have completed')

### Before vs After Comparison
Side-by-side comparison of model performance before and after training.

In [ ]:
comparison_path = './training_output/comparison.png'
if os.path.exists(comparison_path):
    display(Image(comparison_path, width=900))
    print('📊 Green = trained model, Gray = baseline (untrained)')
else:
    print('⚠️ Comparison plot not found')

### Detailed Metrics

In [ ]:
import json

results_path = './training_output/comparison_results.json'
if os.path.exists(results_path):
    with open(results_path) as f:
        results = json.load(f)
    
    print('=' * 60)
    print('📊 TRAINING RESULTS SUMMARY')
    print('=' * 60)
    
    b = results['baseline']
    t = results['trained']
    imp = results['improvement']
    
    print(f"\n{'Metric':<25} {'Baseline':>12} {'Trained':>12} {'Delta':>12}")
    print('-' * 60)
    print(f"{'Avg Reward':<25} {b['avg_reward']:>12.4f} {t['avg_reward']:>12.4f} {imp['avg_reward_delta']:>+12.4f}")
    print(f"{'JSON Parse Rate':<25} {b['parse_rate']:>12.1%} {t['parse_rate']:>12.1%} {imp['parse_rate_delta']:>+12.4f}")
    
    print(f"\n✅ Reward improved by {imp['avg_reward_delta']:+.4f}")
else:
    print('⚠️ Results file not found — run training first')

## 6️⃣ Test the Trained Model
Let's see the trained model triage a real email.

In [ ]:
from unsloth import FastLanguageModel
import json

# Load the trained model
model_path = './training_output/model'
if os.path.exists(model_path):
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_path, max_seq_length=2048, load_in_4bit=True
    )
    FastLanguageModel.for_inference(model)
    
    # Test with a sample email
    test_email = """Subject: URGENT - Unauthorized charge on my credit card
From: enterprise_client@megacorp.com (Tier: enterprise)
Body: I noticed a charge of $4,999 on my account that I did not authorize. 
My account also appears to be locked. Please investigate immediately and 
block any further transactions.

Respond with a JSON object containing: priority, department, escalate (boolean), and response."""
    
    messages = [
        {'role': 'system', 'content': 'You are an expert customer support triage agent. Output valid JSON with priority, department, escalate, and response fields.'},
        {'role': 'user', 'content': test_email}
    ]
    
    inputs = tokenizer.apply_chat_template(messages, return_tensors='pt', add_generation_prompt=True).to(model.device)
    outputs = model.generate(inputs, max_new_tokens=256, temperature=0.1, do_sample=True)
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    
    print('🤖 Trained Model Output:')
    print(response)
else:
    print('⚠️ Trained model not found — run training first')

---

## Summary

| What | Details |
|------|--------|
| **Environment** | OmniTriageEnv — 30 emails across 3 difficulty levels |
| **Model** | Llama-3.2-1B-Instruct (4-bit LoRA via Unsloth) |
| **Training** | GRPO with 200 steps, batch size 4, 4 generations per prompt |
| **Reward** | Dense, multi-component (priority + department + escalation + response keywords) |
| **Output** | Reward curve, before/after comparison, saved LoRA weights |

🔗 **Environment:** [HuggingFace Space](https://huggingface.co/spaces/Prakhar132/email-triage-env)  
🔗 **Dashboard:** [Live Demo](https://prakhar132-email-triage-env.hf.space/dashboard)  
🔗 **GitHub:** [Repository](https://github.com/MANu13151/OmniTriageEnv)